# 04 — Analytical Data Model

**Tujuan notebook ini (Fase 4 roadmap):**
Bangun 3 analytical tables terpisah — bukan satu dataframe monster:
- `customer_analytics_full`
- `product_analytics`
- `seller_analytics`

> ⚠️ **Semua tabel di notebook ini dihitung dari SELURUH histori dataset.** Boleh dipakai untuk business reporting/EDA (Fase 5) dan Power BI (Fase 12). **Tidak boleh dipakai langsung sebagai feature ML** — versi ML (`customer_features_observation`, dipotong sampai observation cutoff) baru dibangun nanti di `07_ml_dataset_design.ipynb`, setelah Decision Gate.


In [7]:
import pandas as pd
import numpy as np
from pymongo import MongoClient
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "olist_db"
OUTPUT_DIR = Path("../data_processed")
OUTPUT_DIR.mkdir(exist_ok=True)

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

COLLECTIONS = [
    "customers_raw", "orders_raw", "order_items_raw", "payments_raw",
    "reviews_raw", "products_raw", "sellers_raw", "geolocation_raw",
    "category_translation_raw",
]

dfs = {}
for col_name in COLLECTIONS:
    dfs[col_name] = pd.DataFrame(list(db[col_name].find({}, {"_id": 0})))

print("Semua collection ter-load.")


Semua collection ter-load.


### Rebuild `orders` yang sudah ditransformasi (ringkas ulang dari notebook 03)

Notebook ini independen dari notebook 03 (supaya bisa dijalankan sendiri tanpa dependency antar-notebook), jadi transformasi dasar diulang di sini secara ringkas.


In [8]:
orders = dfs["orders_raw"].copy()

date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days
orders["is_delivered"] = orders["order_delivered_customer_date"].notnull()

payments = dfs["payments_raw"].copy()
transaction_value_per_order = payments.groupby("order_id")["payment_value"].sum().rename("transaction_value")
orders = orders.merge(transaction_value_per_order, on="order_id", how="left")

order_items = dfs["order_items_raw"].copy()
items_per_order = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
).reset_index()
orders = orders.merge(items_per_order, on="order_id", how="left")

# customer_unique_id perlu di-attach ke orders untuk agregasi customer-level
customers = dfs["customers_raw"].copy()
orders = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")

print("orders siap dipakai, shape:", orders.shape)


orders siap dipakai, shape: (99441, 13)


---
## `customer_analytics_full`

Key: `customer_unique_id`. Dihitung dari seluruh histori dataset.


In [9]:
customer_analytics_full = orders.groupby("customer_unique_id").agg(
    total_orders=("order_id", "nunique"),
    total_spending=("transaction_value", "sum"),
    avg_order_value=("transaction_value", "mean"),
    first_order_date=("order_purchase_timestamp", "min"),
    last_order_date=("order_purchase_timestamp", "max"),
).reset_index()

customer_analytics_full["customer_lifetime_days"] = (
    customer_analytics_full["last_order_date"] - customer_analytics_full["first_order_date"]
).dt.days

# customer_state: ambil dari customers_raw (state customer, bukan state order)
customer_state = customers[["customer_unique_id", "customer_state"]].drop_duplicates(subset="customer_unique_id")
customer_analytics_full = customer_analytics_full.merge(customer_state, on="customer_unique_id", how="left")

print("customer_analytics_full shape:", customer_analytics_full.shape)
customer_analytics_full.head()


customer_analytics_full shape: (96096, 8)


,customer_unique_id,total_orders,total_spending,avg_order_value,first_order_date,last_order_date,customer_lifetime_days,customer_state
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,0,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,0,SP
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,0,SC
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,0,PA
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,0,SP


---
## `product_analytics`

Key: `product_id`. Pakai `transaction_value` (bukan `revenue`) — konsisten dengan Fase 3.3.


In [10]:
order_items_full = dfs["order_items_raw"].merge(
    orders[["order_id", "order_status"]], on="order_id", how="left"
)

reviews = dfs["reviews_raw"].copy()
review_per_order = reviews.groupby("order_id")["review_score"].mean().rename("review_score")
order_items_full = order_items_full.merge(review_per_order, on="order_id", how="left")

products = dfs["products_raw"].copy()
category_translation = dfs["category_translation_raw"].copy()
products = products.merge(category_translation, on="product_category_name", how="left")

product_analytics = order_items_full.groupby("product_id").agg(
    total_orders=("order_id", "nunique"),
    units_sold=("order_item_id", "count"),
    transaction_value=("price", "sum"),
    avg_price=("price", "mean"),
    avg_review_score=("review_score", "mean"),
).reset_index()

product_analytics = product_analytics.merge(
    products[["product_id", "product_category_name_english"]], on="product_id", how="left"
).rename(columns={"product_category_name_english": "category"})

print("product_analytics shape:", product_analytics.shape)
product_analytics.head()


product_analytics shape: (32951, 7)


,product_id,total_orders,units_sold,transaction_value,avg_price,avg_review_score,category
0,00066f42aeeb9f3007548bb9d3f33c38,1,1,101.65,101.65,5.0,perfumery
1,00088930e925c41fd95ebfe695fd2655,1,1,129.90,129.90,4.0,auto
2,0009406fd7479715e4bef61dd91f2462,1,1,229.00,229.00,1.0,bed_bath_table
3,000b8f95fcb9e0096488278317764d19,2,2,117.80,58.90,5.0,housewares
4,000d9be29b5207b54e86aa1b1ac54872,1,1,199.00,199.00,5.0,watches_gifts


---
## `seller_analytics`

Key: `seller_id`.


In [11]:
order_items_seller = dfs["order_items_raw"].merge(
    orders[["order_id", "delivery_days", "is_delivered"]], on="order_id", how="left"
).merge(review_per_order, on="order_id", how="left")

sellers = dfs["sellers_raw"].copy()

seller_analytics = order_items_seller.groupby("seller_id").agg(
    orders=("order_id", "nunique"),
    transaction_value=("price", "sum"),
    avg_delivery_days=("delivery_days", "mean"),
    avg_review_score=("review_score", "mean"),
).reset_index()

seller_analytics = seller_analytics.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left"
)

print("seller_analytics shape:", seller_analytics.shape)
seller_analytics.head()


seller_analytics shape: (3095, 6)


,seller_id,orders,transaction_value,avg_delivery_days,avg_review_score,seller_state
0,0015a82c2db000af6aaaf3ae2ecb0532,3,2685.00,10.333333,3.666667,SP
1,001cca7ae9ae17fb1caed9dfb1094831,200,25080.03,12.628205,3.902542,ES
2,001e6ad469a905060d959994f1b41e4f,1,250.00,NaN,1.000000,RJ
3,002100f778ceb8431b7a1020ff7ab48f,51,1234.50,15.777778,3.981818,SP
4,003554e2dce176b5555353e4f3555ac8,1,120.00,4.000000,5.000000,GO


---
## Ringkasan & Simpan ke `data_processed/`

Disimpan sebagai CSV supaya bisa dipakai lagi di notebook 05 (EDA) dan nanti diimport ke Power BI (Fase 12, Opsi A).


In [12]:
print(f"customer_analytics_full : {customer_analytics_full.shape[0]} customer")
print(f"product_analytics        : {product_analytics.shape[0]} produk")
print(f"seller_analytics         : {seller_analytics.shape[0]} seller")

customer_analytics_full.to_csv(OUTPUT_DIR / "customer_analytics.csv", index=False)
product_analytics.to_csv(OUTPUT_DIR / "product_analytics.csv", index=False)
seller_analytics.to_csv(OUTPUT_DIR / "seller_analytics.csv", index=False)

print(f"\n3 file tersimpan di {OUTPUT_DIR.resolve()}")


customer_analytics_full : 96096 customer
product_analytics        : 32951 produk
seller_analytics         : 3095 seller

3 file tersimpan di C:\Users\ahmad farid\olist-end-to-end-data-science\data_processed


---
## Percabangan Setelah Analytical Tables (baca sebelum lanjut ke Fase 5-7)

Sesuai roadmap Fase 4.1 — dari titik ini, alurnya sengaja bercabang dua:

```
                 Analytical Tables (notebook ini)
                       │
            ┌──────────┴──────────┐
            ↓                     ↓
    Business Analytics       ML Dataset Design
            ↓                     ↓
   customer_analytics_full   Target Definition
       (notebook 05, 06)          ↓
            ↓              Observation Window
         Power BI            (notebook 07)
```

Jalur kiri (notebook 05 EDA, 06 segmentation) pakai `customer_analytics_full` apa adanya. Jalur kanan (notebook 07 ML dataset design) akan membangun ulang feature dari `orders` yang dipotong sampai observation cutoff — **bukan** dari tabel `customer_analytics_full` ini.


---
## Definition of Done (Fase 4)

- [ ] 3 analytical tables terbentuk terpisah (bukan 1 dataframe monster)
- [ ] `customer_analytics_full` pakai `customer_unique_id` sebagai key
- [ ] Kolom transaction value pakai nama `transaction_value`, bukan `revenue`
- [ ] Ketiga tabel tersimpan di `data_processed/` sebagai CSV
- [ ] Paham bahwa tabel ini untuk business analytics, BUKAN feature ML langsung

**Lanjut ke:** `05_eda_business_analytics.ipynb`
